In [3]:
print("Question 3")

Question 3


In [4]:
import numpy as np
from sklearn.datasets import fetch_olivetti_faces

seed = 26006
rng = np.random.default_rng(seed)

faces = fetch_olivetti_faces()
X = faces.data
y = faces.target
classes = np.arange(40)

train, validate, test = [], [], []

for person in range(40):
    index = rng.permutation(np.where(y == person)[0])
    train += list(index[:6])
    validate += list(index[6:8])
    test += list(index[8:])

X_train = X[train]
y_train = y[train]
X_test = X[test]
y_test = y[test]

In [5]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

TREE_DEPTH = None
MIN_SAMPLES_LEAF = 4
B_FINAL = 50
BOOST_DEPTH = 3
BOOST_ROUNDS = 100
R = 30

def resample_training_set(rng):
    selected_indices = []
    for identity in classes:
        identity_positions = np.where(y_train == identity)[0]
        sampled_positions = rng.choice(identity_positions, size=6, replace=True)
        selected_indices.extend(sampled_positions)
    selected_indices = np.array(selected_indices)
    rng.shuffle(selected_indices)
    return X_train[selected_indices], y_train[selected_indices]

def single_tree_vectors(model, X_eval):
    probabilities = model.predict_proba(X_eval)
    vectors = np.zeros((len(X_eval), 40))
    for j, class_label in enumerate(model.classes_):
        vectors[:, int(class_label)] = probabilities[:, j]
    return vectors

def bagging_vectors(trees, X_eval):
    predictions = np.array([tree.predict(X_eval) for tree in trees])
    vectors = np.zeros((len(X_eval), 40))
    for tree_predictions in predictions:
        for i, label in enumerate(tree_predictions):
            vectors[i, int(label)] += 1
    vectors /= len(trees)
    return vectors

def boosting_vectors(model, X_eval):
    vectors = np.zeros((len(X_eval), 40))
    total_alpha = np.sum(model["alphas"])
    for tree, alpha in zip(model["learners"], model["alphas"]):
        predictions = tree.predict(X_eval)
        for i, label in enumerate(predictions):
            vectors[i, int(label)] += alpha
    if total_alpha > 0:
        vectors /= total_alpha
    return vectors

In [6]:
print("Part (a)")

Part (a)


In [13]:
def train_samme(x_train, y_train, depth, num_rounds):
    N = len(x_train)
    K = 40 
    w = np.ones(N) / N

    models = []
    alphas = []

    track_errors = []
    track_alphas = []
    track_max_w = []
    track_top10_w = []

    for t in range(num_rounds):
        # train week learner 
        tree = DecisionTreeClassifier(max_depth=depth, random_state=seed)
        tree.fit(x_train, y_train, sample_weight=w)
        preds = tree.predict(x_train)

        # calculate the weighted error 
        incorrect = (preds != y_train)
        err = np.sum(w[incorrect]) / np.sum(w)

        # stop if the learner is wrose than random guessing 
        if err >= (K - 1) / K:
            break 

        # calculate alpha (learner weight)
        alpha = np.log((1 - err) / err) + np.log(K - 1)

        # save metrics for plotting 
        track_errors.append(err)
        track_alphas.append(alpha)
        track_max_w.append(np.max(w))

        # sort the weights in descending order and sum the top 10 
        track_top10_w.append(np.sum(np.sort(w)[::-1][:10]))

        # update and normalize weights
        w = w * np.exp(alpha * incorrect)
        w = w / np.sum(w)

        # save the tree and its weight
        models.append(tree)
        alphas.append(alpha)
    
    return models, alphas, track_errors, track_alphas, track_max_w, track_top10_w

In [14]:
q3_rng = np.random.default_rng(seed)
models = {"Single Tree": [], "Bagging (50 trees)": [], "SAMME Boosting": []}
prediction_history = {name: [] for name in models}
accuracy_history = {name: [] for name in models}

for r in range(R):
    X_resampled, y_resampled = resample_training_set(q3_rng)
    repetition_seed = seed + 1000 * (r + 1)

    # 1. Single
    single_tree = DecisionTreeClassifier(max_depth=TREE_DEPTH, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=repetition_seed)
    single_tree.fit(X_resampled, y_resampled)

    # 2. Bagging
    bag_rng = np.random.default_rng(repetition_seed)
    bagged_trees = []
    for b in range(B_FINAL):
        bootstrap_indices = bag_rng.choice(len(X_resampled), size=len(X_resampled), replace=True)
        tree = DecisionTreeClassifier(max_depth=TREE_DEPTH, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=repetition_seed + b + 1)
        tree.fit(X_resampled[bootstrap_indices], y_resampled[bootstrap_indices])
        bagged_trees.append(tree)

    # 3. SAMME Boosting 
    learners, alphas, _, _, _, _ = train_samme(
        X_resampled, 
        y_resampled, 
        BOOST_DEPTH, 
        BOOST_ROUNDS
    )

    boost_model = {
        "learners": learners,
        "alphas": np.asarray(alphas)
    }

    # Store predictions
    repetition_predictions = {
        "Single Tree": single_tree_vectors(single_tree, X_test),
        "Bagging (50 trees)": bagging_vectors(bagged_trees, X_test),
        "SAMME Boosting": boosting_vectors(boost_model, X_test)
    }

    for name, vectors in repetition_predictions.items():
        prediction_history[name].append(vectors)
        predicted_labels = classes[np.argmax(vectors, axis=1)]
        accuracy_history[name].append(np.mean(predicted_labels == y_test))

# Calculate and display
one_hot_targets = np.eye(40)[y_test]
results = []

for name in models:
    predictions = np.asarray(prediction_history[name])
    mean_predictions = np.mean(predictions, axis=0)
    
    bias_squared = np.mean(np.sum((mean_predictions - one_hot_targets) ** 2, axis=1))
    variance = np.mean(np.sum((predictions - mean_predictions[None, :, :]) ** 2, axis=2))
    accuracies = np.asarray(accuracy_history[name])

    results.append({
        "Model": name,
        "Bias Squared": bias_squared,
        "Variance": variance,
        "Mean Test Accuracy": np.mean(accuracies),
        "Std Test Accuracy": np.std(accuracies, ddof=0)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Model,Bias Squared,Variance,Mean Test Accuracy,Std Test Accuracy
0,Single Tree,0.3988,0.6421,0.4796,0.0484
1,Bagging (50 trees),0.5378,0.0326,0.7250,0.0399
2,SAMME Boosting,0.9085,0.0158,0.2417,0.0726


In [9]:
print("part (b)")

part (b)


In [16]:
# Set R_B to 5 or 10 for a quicker test run, or 30 for the final assignment
R_B = 5 

depth_settings = {"Shallow (depth=2)": 2, "Selected (unrestricted)": None, "Deep (depth=20)": 20}
q3b_rng = np.random.default_rng(seed + 30000)
q3b_names = [f"{model} - {depth}" for depth in depth_settings for model in ["Single Tree", "Bagging 100"]]
q3b_predictions = {name: [] for name in q3b_names}
q3b_accuracies = {name: [] for name in q3b_names}

for r in range(R_B):
    print(f"Running repetition {r + 1}/{R_B}...")
    X_resampled, y_resampled = resample_training_set(q3b_rng)
    repetition_seed = seed + 40000 + r * 1000

    for depth_name, depth in depth_settings.items():
        # Single Tree
        single_tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=repetition_seed)
        single_tree.fit(X_resampled, y_resampled)
        single_vec = single_tree_vectors(single_tree, X_test)
        
        q3b_predictions[f"Single Tree - {depth_name}"].append(single_vec)
        q3b_accuracies[f"Single Tree - {depth_name}"].append(np.mean(classes[np.argmax(single_vec, axis=1)] == y_test))

        # Bagging (100 trees)
        bag_rng = np.random.default_rng(repetition_seed + 100)
        bagged_trees = []
        for b in range(100):
            bootstrap_indices = bag_rng.choice(len(X_resampled), size=len(X_resampled), replace=True)
            tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=repetition_seed + b + 101)
            tree.fit(X_resampled[bootstrap_indices], y_resampled[bootstrap_indices])
            bagged_trees.append(tree)

        bag_vec = bagging_vectors(bagged_trees, X_test)
        q3b_predictions[f"Bagging 100 - {depth_name}"].append(bag_vec)
        q3b_accuracies[f"Bagging 100 - {depth_name}"].append(np.mean(classes[np.argmax(bag_vec, axis=1)] == y_test))

# Calculate metrics for Q3(b)
q3b_results = []
for name in q3b_names:
    pred = np.asarray(q3b_predictions[name])
    mean_pred = np.mean(pred, axis=0)
    
    bias_squared = np.mean(np.sum((mean_pred - one_hot_targets) ** 2, axis=1))
    variance = np.mean(np.sum((pred - mean_pred[None, :, :]) ** 2, axis=2))
    accuracies = np.asarray(q3b_accuracies[name])

    q3b_results.append({
        "Model": name,
        "Bias squared": bias_squared,
        "Variance": variance,
        "Mean test accuracy": np.mean(accuracies),
        "SD test accuracy": np.std(accuracies, ddof=0)
    })

q3b_table = pd.DataFrame(q3b_results)
display(q3b_table.round(4))

Running repetition 1/5...
Running repetition 2/5...
Running repetition 3/5...
Running repetition 4/5...
Running repetition 5/5...


,Model,Bias squared,Variance,Mean test accuracy,SD test accuracy
0,Single Tree - Shallow (depth=2),0.9217,0.0339,0.0650,0.0146
1,Bagging 100 - Shallow (depth=2),0.9323,0.0079,0.1775,0.0544
2,Single Tree - Selected (unrestricted),0.5510,0.5390,0.4550,0.0710
3,Bagging 100 - Selected (unrestricted),0.5424,0.0190,0.7400,0.0320
4,Single Tree - Deep (depth=20),0.6578,0.2475,0.3200,0.0292
5,Bagging 100 - Deep (depth=20),0.6232,0.0165,0.6875,0.0209


In [17]:
print("Part C")

Part C


In [ ]:
# Part (c): Depth comparison, leaf occupancy, and disagreement
B_C = 100
RANDOM_SEED_C = seed + 50000
q3c_rng = np.random.default_rng(RANDOM_SEED_C)

# Use one reproducible resampled training set for this analysis
print("Resampling training set for Part (c)...")
X_resampled, y_resampled = resample_training_set(q3c_rng)
q3c_rows = []

for depth_name, depth in depth_settings.items():
    print(f"Processing depth setting: {depth_name}...")
    
    # 1. Single Tree Leaf Occupancy
    single_tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=RANDOM_SEED_C)
    single_tree.fit(X_resampled, y_resampled)
    
    single_leaf_ids = np.where(single_tree.tree_.children_left == -1)[0]
    single_leaf_counts = single_tree.tree_.n_node_samples[single_leaf_ids]
    single_mean_occupancy = np.mean(single_leaf_counts)
    single_singleton_fraction = np.mean(single_leaf_counts == 1)

    # 2. Bagged Ensemble Leaf Occupancy & Training
    bag_rng = np.random.default_rng(RANDOM_SEED_C + 100)
    bagged_trees = []
    bag_leaf_occupancies = []
    bag_singleton_fractions = []

    for b in range(B_C):
        indices = bag_rng.choice(len(X_resampled), size=len(X_resampled), replace=True)
        tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=RANDOM_SEED_C + b + 101)
        tree.fit(X_resampled[indices], y_resampled[indices])
        bagged_trees.append(tree)

        leaf_ids = np.where(tree.tree_.children_left == -1)[0]
        leaf_counts = tree.tree_.n_node_samples[leaf_ids]
        bag_leaf_occupancies.append(np.mean(leaf_counts))
        bag_singleton_fractions.append(np.mean(leaf_counts == 1))

    # 3. Pairwise Disagreement on Clean Test Set
    predictions = np.array([tree.predict(X_test) for tree in bagged_trees])
    disagreements = []

    for i in range(len(X_test)):
        _, counts = np.unique(predictions[:, i], return_counts=True)
        agreeing_pairs = np.sum(counts * (counts - 1)) / 2
        total_pairs = B_C * (B_C - 1) / 2
        disagreements.append(1 - agreeing_pairs / total_pairs)
    
    mean_disagreement = np.mean(disagreements)

    # 4. Retrieve accuracy/variance metrics from Part (b) table
    single_row = q3b_table[q3b_table["Model"] == f"Single Tree - {depth_name}"].iloc[0]
    bag_row = q3b_table[q3b_table["Model"] == f"Bagging 100 - {depth_name}"].iloc[0]

    single_acc = single_row["Mean test accuracy"]
    bag_acc = bag_row["Mean test accuracy"]
    single_var = single_row["Variance"]
    bag_var = bag_row["Variance"]

    q3c_rows.append({
        "Depth": depth_name,
        "Single accuracy": single_acc,
        "Bagged accuracy": bag_acc,
        "Accuracy improvement": bag_acc - single_acc,
        "Single bias^2": single_row["Bias squared"],
        "Bagged bias^2": bag_row["Bias squared"],
        "Single variance": single_var,
        "Bagged variance": bag_var,
        "Variance reduction": single_var - bag_var,
        "Single mean leaf occupancy": single_mean_occupancy,
        "Single singleton leaf fraction": single_singleton_fraction,
        "Bagged mean leaf occupancy": np.mean(bag_leaf_occupancies),
        "Bagged singleton leaf fraction": np.mean(bag_singleton_fractions),
        "Pairwise disagreement": mean_disagreement
    })

q3c_table = pd.DataFrame(q3c_rows)
display(q3c_table.round(4))

# 5. Output Conclusion
best_improvement = q3c_table.loc[q3c_table["Accuracy improvement"].idxmax()]
deepest = q3c_table.iloc[-1]

print("\nLargest accuracy improvement from bagging:", best_improvement["Depth"])
print("Deepest tree:", deepest["Depth"])

if best_improvement["Depth"] == deepest["Depth"]:
    print("The deepest tree HAD the largest accuracy improvement.")
else:
    print("The deepest tree did NOT have the largest accuracy improvement.")

Resampling training set for Part (c)...
Processing depth setting: Shallow (depth=2)...
Processing depth setting: Selected (unrestricted)...
Processing depth setting: Deep (depth=20)...

Generating final table...


,Depth,Single accuracy,Bagged accuracy,Accuracy improvement,Single bias^2,Bagged bias^2,Single variance,Bagged variance,Variance reduction,Single mean leaf occupancy,Single singleton leaf fraction,Bagged mean leaf occupancy,Bagged singleton leaf fraction,Pairwise disagreement
0,Shallow (depth=2),0.065,0.1775,0.1125,0.9217,0.9323,0.0339,0.0079,0.026,80.0000,0.0,77.2000,0.0,0.9545
1,Selected (unrestricted),0.455,0.7400,0.2850,0.5510,0.5424,0.5390,0.0190,0.520,6.0000,0.0,6.1119,0.0,0.8278
2,Deep (depth=20),0.320,0.6875,0.3675,0.6578,0.6232,0.2475,0.0165,0.231,11.4286,0.0,9.2929,0.0,0.8771



Largest accuracy improvement from bagging: Deep (depth=20)
Deepest tree: Deep (depth=20)
Conclusion: The deepest tree HAD the largest accuracy improvement.


In [20]:
# Part (d): Final comparison table with Clean/Masked Accuracy, Bias, and Variance

# 1. Create the masked test set (masking the bottom half of the images)
X_test_masked = X_test.copy()
for i in range(len(X_test_masked)):
    img = X_test_masked[i].reshape(64, 64)
    img[32:, :] = 0  
    X_test_masked[i] = img.flatten()

# 2. Train the selected models on the original full training set
# Single Tree
final_tree = DecisionTreeClassifier(max_depth=TREE_DEPTH, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=seed)
final_tree.fit(X_train, y_train)

# Bagging (50 trees)
final_bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=TREE_DEPTH, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=seed),
    n_estimators=B_FINAL,
    random_state=seed
)
final_bagging.fit(X_train, y_train)

# SAMME Boosting (100 rounds, depth 3)
learners, alphas, _, _, _, _ = train_samme(X_train, y_train, BOOST_DEPTH, BOOST_ROUNDS)
final_boost = {"learners": learners, "alphas": np.asarray(alphas)}

# 3. Evaluate Clean and Masked Accuracies
def get_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

tree_clean_acc = get_accuracy(y_test, final_tree.predict(X_test))
tree_mask_acc = get_accuracy(y_test, final_tree.predict(X_test_masked))

bag_clean_acc = get_accuracy(y_test, final_bagging.predict(X_test))
bag_mask_acc = get_accuracy(y_test, final_bagging.predict(X_test_masked))

boost_clean_preds = classes[np.argmax(boosting_vectors(final_boost, X_test), axis=1)]
boost_mask_preds = classes[np.argmax(boosting_vectors(final_boost, X_test_masked), axis=1)]
boost_clean_acc = get_accuracy(y_test, boost_clean_preds)
boost_mask_acc = get_accuracy(y_test, boost_mask_preds)

# 4. Assemble the Final Table using Bias and Variance from Part A (results_df)
final_rows = [
    {
        "Model": "Single Tree",
        "Clean Test Accuracy": tree_clean_acc,
        "Masked Test Accuracy": tree_mask_acc,
        "Bias Squared": results_df.loc[results_df["Model"] == "Single Tree", "Bias Squared"].values[0],
        "Variance": results_df.loc[results_df["Model"] == "Single Tree", "Variance"].values[0]
    },
    {
        "Model": "Bagging (50 trees)",
        "Clean Test Accuracy": bag_clean_acc,
        "Masked Test Accuracy": bag_mask_acc,
        "Bias Squared": results_df.loc[results_df["Model"] == "Bagging (50 trees)", "Bias Squared"].values[0],
        "Variance": results_df.loc[results_df["Model"] == "Bagging (50 trees)", "Variance"].values[0]
    },
    {
        "Model": "SAMME Boosting",
        "Clean Test Accuracy": boost_clean_acc,
        "Masked Test Accuracy": boost_mask_acc,
        "Bias Squared": results_df.loc[results_df["Model"] == "SAMME Boosting", "Bias Squared"].values[0],
        "Variance": results_df.loc[results_df["Model"] == "SAMME Boosting", "Variance"].values[0]
    }
]

final_table = pd.DataFrame(final_rows)
print("\nQ3(d): FINAL MODEL COMPARISON")
display(final_table.round(6))

final_table.to_csv("Q3d_final_comparison.csv", index=False)
print("Table saved as Q3d_final_comparison.csv")


Q3(d): FINAL MODEL COMPARISON


,Model,Clean Test Accuracy,Masked Test Accuracy,Bias Squared,Variance
0,Single Tree,0.6250,0.0500,0.398787,0.642080
1,Bagging (50 trees),0.8000,0.1750,0.537814,0.032625
2,SAMME Boosting,0.3125,0.0875,0.908503,0.015845


Table saved as Q3d_final_comparison.csv
